# Training: DeBERTa-v3-base for Sentiment Classification
##### Ömer Faruk Merey - Middle East Technical University

Fine-tuning `microsoft/deberta-v3-base` on the BrownBox customer service conversation dataset.

**Key decisions:**
- Head+tail truncation (first 256 + last 256 tokens) to handle long conversations within the 512 token limit
- Weighted cross-entropy loss to address severe class imbalance (positive class is only 1.7%)
- Experiment tracking with Weights & Biases (wandb)

## 1. Setup & Load Data

In [ ]:
import pandas as pd
import numpy as np
import json
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
import matplotlib.pyplot as plt
import wandb

device = torch.device('mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

In [ ]:
# Load preprocessed data
train_df = pd.read_csv('../dataset/processed/train_split.csv')
val_df = pd.read_csv('../dataset/processed/val_split.csv')
test_df = pd.read_csv('../dataset/processed/test_processed.csv')

with open('../dataset/processed/metadata.json') as f:
    metadata = json.load(f)

label_map = metadata['label_map']
label_map_inv = {v: k for k, v in label_map.items()}
class_weights = torch.tensor(metadata['class_weights'], dtype=torch.float32).to(device)

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")
print(f"Labels: {label_map}")
print(f"Class weights: {metadata['class_weights']}")

## 2. Tokenization with Head+Tail Truncation

Since ~50% of conversations exceed DeBERTa's 512 token limit, simple truncation loses the end of conversations where sentiment resolution often occurs.

**Strategy:** Take the first 256 tokens (problem statement) + last 256 tokens (resolution/outcome), joined with a `[SEP]` token.

In [ ]:
MODEL_NAME = 'microsoft/deberta-v3-base'
MAX_LENGTH = 512
HEAD_TOKENS = 256
TAIL_TOKENS = 254  # 256 - 2 for [CLS] and [SEP]

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Tokenizer loaded: {MODEL_NAME}")

In [ ]:
def head_tail_tokenize(texts, tokenizer, max_length=MAX_LENGTH, head_tokens=HEAD_TOKENS, tail_tokens=TAIL_TOKENS):
    """Tokenize with head+tail strategy for long texts."""
    all_input_ids = []
    all_attention_masks = []

    for text in texts:
        tokens = tokenizer.encode(text, add_special_tokens=False)

        if len(tokens) <= max_length - 2:  # fits within limit (with [CLS] and [SEP])
            encoding = tokenizer(
                text,
                max_length=max_length,
                padding='max_length',
                truncation=True,
                return_tensors='pt'
            )
        else:
            # Head + tail truncation
            head = tokens[:head_tokens]
            tail = tokens[-tail_tokens:]
            combined = [tokenizer.cls_token_id] + head + [tokenizer.sep_token_id] + tail + [tokenizer.sep_token_id]

            # Pad if needed
            pad_length = max_length - len(combined)
            attention_mask = [1] * len(combined) + [0] * pad_length
            combined = combined + [tokenizer.pad_token_id] * pad_length

            encoding = {
                'input_ids': torch.tensor([combined]),
                'attention_mask': torch.tensor([attention_mask])
            }

        all_input_ids.append(encoding['input_ids'].squeeze(0))
        all_attention_masks.append(encoding['attention_mask'].squeeze(0))

    return {
        'input_ids': torch.stack(all_input_ids),
        'attention_mask': torch.stack(all_attention_masks)
    }

print("Tokenizing train set...")
train_encodings = head_tail_tokenize(train_df['conversation'].tolist(), tokenizer)
print("Tokenizing val set...")
val_encodings = head_tail_tokenize(val_df['conversation'].tolist(), tokenizer)
print("Tokenizing test set...")
test_encodings = head_tail_tokenize(test_df['conversation'].tolist(), tokenizer)

print(f"\nTrain: {train_encodings['input_ids'].shape}")
print(f"Val:   {val_encodings['input_ids'].shape}")
print(f"Test:  {test_encodings['input_ids'].shape}")

## 3. Dataset & DataLoader

In [ ]:
class SentimentDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = torch.tensor(labels.values, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels': self.labels[idx]
        }

BATCH_SIZE = 8

train_dataset = SentimentDataset(train_encodings, train_df['label'])
val_dataset = SentimentDataset(val_encodings, val_df['label'])
test_dataset = SentimentDataset(test_encodings, test_df['label'])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")

## 4. Model Setup

In [ ]:
NUM_LABELS = 3
LEARNING_RATE = 2e-5
NUM_EPOCHS = 5
WARMUP_RATIO = 0.1

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS
)
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

total_steps = len(train_loader) * NUM_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

criterion = nn.CrossEntropyLoss(weight=class_weights)

print(f"Model: {MODEL_NAME}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable:  {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"Epochs: {NUM_EPOCHS}, LR: {LEARNING_RATE}, Batch size: {BATCH_SIZE}")
print(f"Total steps: {total_steps}, Warmup steps: {warmup_steps}")

## 5. Training Loop with W&B Logging

In [ ]:
wandb.init(
    project='customer-sentiment-analysis',
    name='deberta-v3-base-head-tail',
    config={
        'model': MODEL_NAME,
        'max_length': MAX_LENGTH,
        'truncation': 'head+tail',
        'head_tokens': HEAD_TOKENS,
        'tail_tokens': TAIL_TOKENS,
        'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE,
        'epochs': NUM_EPOCHS,
        'warmup_ratio': WARMUP_RATIO,
        'class_weights': metadata['class_weights'],
        'train_size': len(train_df),
        'val_size': len(val_df),
        'test_size': len(test_df)
    }
)

In [ ]:
def evaluate(model, loader, criterion, device):
    """Evaluate model on a data loader. Returns loss, preds, and labels."""
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(outputs.logits, labels)
            total_loss += loss.item()

            preds = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    return avg_loss, np.array(all_preds), np.array(all_labels)

In [ ]:
best_val_f1 = 0
best_model_state = None

for epoch in range(NUM_EPOCHS):
    # Training
    model.train()
    total_train_loss = 0

    for step, batch in enumerate(train_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = criterion(outputs.logits, labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_train_loss += loss.item()

        if (step + 1) % 20 == 0:
            wandb.log({
                'train/step_loss': loss.item(),
                'train/learning_rate': scheduler.get_last_lr()[0]
            })

    avg_train_loss = total_train_loss / len(train_loader)

    # Validation
    val_loss, val_preds, val_labels = evaluate(model, val_loader, criterion, device)
    val_f1 = f1_score(val_labels, val_preds, average='weighted')
    val_acc = accuracy_score(val_labels, val_preds)
    val_f1_macro = f1_score(val_labels, val_preds, average='macro')

    wandb.log({
        'epoch': epoch + 1,
        'train/loss': avg_train_loss,
        'val/loss': val_loss,
        'val/accuracy': val_acc,
        'val/f1_weighted': val_f1,
        'val/f1_macro': val_f1_macro
    })

    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | "
          f"Train Loss: {avg_train_loss:.4f} | "
          f"Val Loss: {val_loss:.4f} | "
          f"Val Acc: {val_acc:.4f} | "
          f"Val F1(w): {val_f1:.4f} | "
          f"Val F1(m): {val_f1_macro:.4f}")

    # Save best model
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_model_state = model.state_dict().copy()
        print(f"  -> New best model (F1: {val_f1:.4f})")

print(f"\nBest validation F1 (weighted): {best_val_f1:.4f}")

## 6. Evaluate on Test Set

In [ ]:
# Load best model
model.load_state_dict(best_model_state)

test_loss, test_preds, test_labels = evaluate(model, test_loader, criterion, device)
test_acc = accuracy_score(test_labels, test_preds)
test_f1_weighted = f1_score(test_labels, test_preds, average='weighted')
test_f1_macro = f1_score(test_labels, test_preds, average='macro')

print(f"Test Loss:        {test_loss:.4f}")
print(f"Test Accuracy:    {test_acc:.4f}")
print(f"Test F1 (weighted): {test_f1_weighted:.4f}")
print(f"Test F1 (macro):   {test_f1_macro:.4f}")

In [ ]:
# Classification report
target_names = [label_map_inv[i] for i in range(NUM_LABELS)]
print(classification_report(test_labels, test_preds, target_names=target_names))

In [ ]:
# Confusion matrix
cm = confusion_matrix(test_labels, test_preds)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
ax.set_title('Confusion Matrix — Test Set')
plt.colorbar(im, ax=ax)

tick_marks = np.arange(NUM_LABELS)
ax.set_xticks(tick_marks)
ax.set_xticklabels(target_names, rotation=45)
ax.set_yticks(tick_marks)
ax.set_yticklabels(target_names)

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black')

ax.set_ylabel('True Label')
ax.set_xlabel('Predicted Label')
plt.tight_layout()
plt.show()

# Log to wandb
wandb.log({
    'test/accuracy': test_acc,
    'test/f1_weighted': test_f1_weighted,
    'test/f1_macro': test_f1_macro,
    'test/loss': test_loss,
    'test/confusion_matrix': wandb.plot.confusion_matrix(
        y_true=test_labels.tolist(),
        preds=test_preds.tolist(),
        class_names=target_names
    )
})

## 7. Save Model

In [ ]:
import os

save_dir = '../models/deberta-v3-sentiment'
os.makedirs(save_dir, exist_ok=True)

model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

# Save training config
config = {
    'model_name': MODEL_NAME,
    'max_length': MAX_LENGTH,
    'head_tokens': HEAD_TOKENS,
    'tail_tokens': TAIL_TOKENS,
    'label_map': label_map,
    'test_accuracy': test_acc,
    'test_f1_weighted': test_f1_weighted,
    'test_f1_macro': test_f1_macro
}
with open(f'{save_dir}/training_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print(f"Model saved to {save_dir}/")
for fname in sorted(os.listdir(save_dir)):
    size = os.path.getsize(f'{save_dir}/{fname}') / (1024*1024)
    print(f"  {fname}: {size:.1f} MB")

In [ ]:
wandb.finish()
print("Done.")